
# EGMA-Math prompt experiments (SmolVLM2-2.2B)

This notebook documents **phase-wise optimizations** for the `egma-math` multiple-choice benchmark (179 trials, Number Line items excluded). Each phase can be toggled in `scripts/experiment_egma_math_phases.py`.

**Artifacts:** per-phase CSVs and logs under `results/egma-math-phases/`.


In [1]:

from __future__ import annotations

import csv
import importlib.util
from pathlib import Path

import pandas as pd

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'data').is_dir() and (p / 'src').is_dir()),
    Path.cwd().parent.parent,
)

RESULTS = ROOT / "results" / "prompt_optimization" / "egma-math" / "qwen-0.8b"
# Load prompt builders from the experiment script
_spec = importlib.util.spec_from_file_location(
    "egma_phases", ROOT / "scripts" / "prompt_optimization" / "egma-math" / "experiment.py"
)
eg = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(eg)

manifest = eg.load_manifest(ROOT / "data")
TRIALS = {t["item_uid"]: t for t in eg.build_trials(manifest)}


def build_prompt(phases: set[int], item_uid: str) -> str:
    """Reconstruct the exact prompt used for a given phase combination."""
    t = TRIALS[item_uid]
    row, opts = t["row"], t["options"]
    if 1 in phases:
        p = eg.build_prompt_phase1(row, opts)
    else:
        p = eg.build_prompt_baseline(row, opts)
    if 4 in phases:
        p = eg.apply_phase4_fewshot(p, t["trial_type"])
    if 5 in phases:
        p = eg.apply_phase5_cot(p, t["trial_type"])
    if 3 in phases:
        p = eg.apply_phase3_system(p)
    return p


def load_result(csv_name: str, item_uid: str) -> dict | None:
    path = RESULTS / csv_name
    if not path.exists():
        return None
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row["item_uid"] == item_uid:
                return row
    return None



## Summary: all runs

Overall accuracy and parse rate (model output could be mapped to A/B/C/D) for each configuration. *Δ vs baseline* is the accuracy difference vs phase 0 in percentage points.


In [2]:

def summarize_all():
    rows = []
    configs = [
        ("baseline", "phase_baseline.csv", "0 — baseline (manifest-style prompts)"),
        ("1", "phase_1.csv", "1 — multiline prompts + block options"),
        ("2", "phase_2.csv", "2 — numeric / value → label parse fallback"),
        ("3", "phase_3.csv", "3 — system prompt + strict answer suffix"),
        ("4", "phase_4.csv", "4 — few-shot examples per trial type"),
        ("5", "phase_5.csv", "5 — chain-of-thought on arithmetic types"),
        ("1_5", "phase_1_5.csv", "1 + 5 combined"),
        ("1_2_3_4_5", "phase_1_2_3_4_5.csv", "1 + 2 + 3 + 4 + 5 combined"),
    ]
    baseline_acc = None
    for cid, fname, desc in configs:
        path = RESULTS / fname
        if not path.exists():
            continue
        n = ok = pr = 0
        with open(path, newline="", encoding="utf-8") as f:
            for r in csv.DictReader(f):
                n += 1
                ok += r.get("is_correct", "").lower() == "true"
                pr += r.get("parsed", "").lower() == "true"
        acc = ok / n if n else 0
        parse_r = pr / n if n else 0
        if cid == "baseline":
            baseline_acc = acc
        delta = "—" if baseline_acc is None or cid == "baseline" else f"{(acc - baseline_acc) * 100:+.1f}"
        rows.append(
            {
                "config": cid,
                "description": desc,
                "n": n,
                "accuracy": f"{acc:.1%}",
                "parse_rate": f"{parse_r:.1%}",
                "delta_pp_vs_baseline": delta,
            }
        )
    return pd.DataFrame(rows)


summary_df = summarize_all()
summary_df


,config,description,n,accuracy,parse_rate,delta_pp_vs_baseline
0,baseline,0 — baseline (manifest-style prompts),179,29.6%,93.9%,—
1,1,1 — multiline prompts + block options,179,50.8%,95.5%,+21.2
2,2,2 — numeric / value → label parse fallback,179,29.6%,93.9%,+0.0
3,3,3 — system prompt + strict answer suffix,179,36.9%,100.0%,+7.3
4,4,4 — few-shot examples per trial type,179,27.9%,98.9%,-1.7
5,5,5 — chain-of-thought on arithmetic types,179,57.0%,81.0%,+27.4
6,1_5,1 + 5 combined,179,53.6%,87.7%,+24.0
7,1_2_3_4_5,1 + 2 + 3 + 4 + 5 combined,179,54.2%,100.0%,+24.6


## Per-trial-type breakdown (all phase CSVs)

The tables below match the console summary from `experiment_egma_math_phases.py` (same ordering: **accuracy** within each table, descending). Columns are **N**, **Acc**, **Parse%**, and **Chance** — with `Chance` using the same heuristic as the script: `0.25` if that trial type has more than 2 items, else `0.5`. The **OVERALL** row is appended at the bottom of each table.


In [3]:
from IPython.display import display, Markdown


def breakdown_by_trial_type(csv_path: Path) -> pd.DataFrame:
    """Replicate run_phase_*.log per-type table from a phase_*.csv file."""
    by_type: dict[str, dict] = {}
    with open(csv_path, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            tt = r["trial_type"]
            row = by_type.setdefault(tt, {"n": 0, "correct": 0, "parsed": 0})
            row["n"] += 1
            row["correct"] += r.get("is_correct", "").lower() == "true"
            row["parsed"] += r.get("parsed", "").lower() == "true"
    rows: list[dict] = []
    for tt, row in sorted(
        by_type.items(), key=lambda kv: kv[1]["correct"] / max(kv[1]["n"], 1), reverse=True
    ):
        n = row["n"]
        acc = row["correct"] / n if n else 0.0
        parse_r = row["parsed"] / n if n else 0.0
        chance = 0.25 if n > 2 else 0.5
        rows.append(
            {
                "Trial Type": tt,
                "N": n,
                "Acc": acc,
                "Parse%": parse_r,
                "Chance": chance,
            }
        )
    total_n = sum(rec["N"] for rec in rows) if rows else 0
    total_ok = sum(by_type[tt]["correct"] for tt in by_type)
    total_parsed = sum(by_type[tt]["parsed"] for tt in by_type)
    overall_acc = total_ok / total_n if total_n else 0.0
    overall_parse = total_parsed / total_n if total_n else 0.0
    rows.append(
        {
            "Trial Type": "OVERALL",
            "N": total_n,
            "Acc": overall_acc,
            "Parse%": overall_parse,
            "Chance": float("nan"),
        }
    )
    return pd.DataFrame(rows)


PHASE_CSVS = [
    ("0 — baseline", "phase_baseline.csv"),
    ("1", "phase_1.csv"),
    ("2", "phase_2.csv"),
    ("3", "phase_3.csv"),
    ("4", "phase_4.csv"),
    ("5", "phase_5.csv"),
    ("1+5", "phase_1_5.csv"),
    ("1+2+3+4+5", "phase_1_2_3_4_5.csv"),
]

for label, fname in PHASE_CSVS:
    path = RESULTS / fname
    if not path.exists():
        continue
    df_bt = breakdown_by_trial_type(path)
    display(Markdown(f"### Phase {label} — `{fname}`"))
    display(
        df_bt.style.format(
            {"Acc": "{:.1%}", "Parse%": "{:.1%}", "Chance": lambda x: "—" if pd.isna(x) else f"{x:.1%}"}
        )
        .hide(axis="index")
    )


### Phase 0 — baseline — `phase_baseline.csv`

Trial Type,N,Acc,Parse%,Chance
Number Comparison,17,52.9%,100.0%,25.0%
Fraction,18,38.9%,100.0%,25.0%
Multiplication,27,37.0%,100.0%,25.0%
Number Identification,26,34.6%,100.0%,25.0%
Missing Number,12,33.3%,100.0%,25.0%
Subtraction,27,29.6%,100.0%,25.0%
Counting AFC,5,20.0%,100.0%,25.0%
Addition,32,15.6%,100.0%,25.0%
Non-symbolic Number Identification,5,0.0%,80.0%,25.0%
Counting,5,0.0%,0.0%,25.0%


### Phase 1 — `phase_1.csv`

Trial Type,N,Acc,Parse%,Chance
Counting,5,100.0%,100.0%,25.0%
Number Identification,26,92.3%,96.2%,25.0%
Counting AFC,5,80.0%,100.0%,25.0%
Non-symbolic Number Comparison,5,80.0%,100.0%,25.0%
Multiplication,27,55.6%,100.0%,25.0%
Number Comparison,17,47.1%,58.8%,25.0%
Subtraction,27,37.0%,100.0%,25.0%
Missing Number,12,33.3%,100.0%,25.0%
Fraction,18,33.3%,100.0%,25.0%
Addition,32,31.2%,100.0%,25.0%


### Phase 2 — `phase_2.csv`

...

Trial Type,N,Acc,Parse%,Chance
Subtraction,27,92.6%,92.6%,25.0%
Addition,32,84.4%,90.6%,25.0%
Multiplication,27,81.5%,81.5%,25.0%
Number Comparison,17,52.9%,100.0%,25.0%
Number Identification,26,34.6%,100.0%,25.0%
Missing Number,12,33.3%,100.0%,25.0%
Fraction,18,27.8%,27.8%,25.0%
Counting AFC,5,20.0%,100.0%,25.0%
Non-symbolic Number Identification,5,0.0%,80.0%,25.0%
Counting,5,0.0%,0.0%,25.0%


### Phase 1+5 — `phase_1_5.csv`

Trial Type,N,Acc,Parse%,Chance
Counting,5,100.0%,100.0%,25.0%
Number Identification,26,92.3%,96.2%,25.0%
Counting AFC,5,80.0%,100.0%,25.0%
Non-symbolic Number Comparison,5,80.0%,100.0%,25.0%
Number Comparison,17,70.6%,82.4%,25.0%
Multiplication,27,59.3%,85.2%,25.0%
Subtraction,27,37.0%,100.0%,25.0%
Addition,32,34.4%,96.9%,25.0%
Missing Number,12,33.3%,100.0%,25.0%
Fraction,18,27.8%,27.8%,25.0%


### Phase 1+2+3+4+5 — `phase_1_2_3_4_5.csv`

Trial Type,N,Acc,Parse%,Chance
Counting,5,100.0%,100.0%,25.0%
Counting AFC,5,100.0%,100.0%,25.0%
Number Identification,26,80.8%,100.0%,25.0%
Non-symbolic Number Comparison,5,80.0%,100.0%,25.0%
Multiplication,27,70.4%,100.0%,25.0%
Number Comparison,17,58.8%,100.0%,25.0%
Missing Number,12,50.0%,100.0%,25.0%
Non-symbolic Number Identification,5,40.0%,100.0%,25.0%
Addition,32,37.5%,100.0%,25.0%
Fraction,18,33.3%,100.0%,25.0%



---

## Phase 0 — Baseline

**What it does:** Uses the `full_prompt` template from `data/assets/manifest.csv` with `<option_i>` substitution. Options are inlined with semicolons (`A: x; B: y; ...`), matching the main `EgmaMathDataset` manifest path.

**Why it matters:** This is the reference format. Some items still say “hear” from the original audio task; the model only sees text.

**Parsing:** Letter-first heuristics (JSON, “answer is X”, leading letter, etc.) — no numeric fallback.



### Example — Number Identification (`math_identify_1`)

**Gold label:** D · **Predicted:** C · **Correct:** False


In [4]:
item_uid = "math_identify_1"
print("--- Prompt (phase 0) ---")
print(build_prompt(set(), item_uid)[:1200])
print()
r = load_result("phase_baseline.csv", item_uid)
print("--- Model output (truncated) ---")
print(r["raw_response"][:800])


--- Prompt (phase 0) ---
Choose the same number: 1. Answer with A, B, C, or D. A: 6; B: 7; C: 9; D: 1

--- Model output (truncated) ---
The answer is C: 6.

Explanation:
The number 1 is a prime number, which means it can only be divided by 1 and itself. The number 1 is also the smallest positive integer. Therefore, the number 1 is equal to 1.



### Example — Number Comparison (`math_compare_8_4`)

**Gold label:** B · **Predicted:** B · **Correct:** True


In [5]:

item_uid = "math_compare_8_4"
print("--- Prompt (phase 0) ---")
print(build_prompt(set(), item_uid)[:1200])
print()
r = load_result("phase_baseline.csv", item_uid)
print("--- Model output (truncated) ---")
print(r["raw_response"][:800])


--- Prompt (phase 0) ---
Choose which is larger. Answer with A or B. A: 4; B: 8

--- Model output (truncated) ---
The answer is B: 8.

Explanation:

The question asks to compare the size of two numbers, 4 and 8. To determine which is larger, we can perform a simple comparison:

1. Compare the first digit of the numbers: 4 is less than 8.



## Phase 1 — Multiline prompts and block options

**What it does:** Replaces the manifest template with a short natural-language stem per `trial_type` and lists options as `A) …` on separate lines, ending with “Answer with one letter.”

**Why it matters:** Dense single-line prompts are harder for small VLMs to parse; spacing and structure improved overall accuracy substantially in our runs.



### Example — Number Identification (`math_identify_1`)

**Gold:** D · **Predicted:** D · **Correct:** True


In [6]:
item_uid = "math_identify_1"
print("--- Prompt (phase 1) ---")
print(build_prompt({1}, item_uid)[:1200])
print()
r = load_result("phase_1.csv", item_uid)
print("--- Model output (truncated) ---")
print(r["raw_response"][:800])


--- Prompt (phase 1) ---
Which number is 1?

A) 6
B) 7
C) 9
D) 1

Answer with one letter.

--- Model output (truncated) ---
The answer is D) 1

Explanation:

The question asks for the number that is equal to 1. The answer is 1, as it is the only number that is equal to itself.



### Example — Addition (`math_add_1_1`)

**Gold:** D · **Predicted:** D · **Correct:** True


In [7]:

item_uid = "math_add_1_1"
print(build_prompt({1}, item_uid)[:1200])
print()
r = load_result("phase_1.csv", item_uid)
print(r["raw_response"][:800])


What is 1+1?

A) 3
B) 0
C) 1
D) 2

Answer with one letter.

The answer is D) 2.

The sum of 1 and 1 is 2. This is a basic arithmetic operation that is often used in mathematics and everyday life.



## Phase 2 — Numeric / value → label parse fallback

**What it does:** Keeps the same prompts as phase 0 (unless combined with other phases). After generation, if the parser cannot extract a letter, it tries to match the **full response** or a quoted value to one of the option strings (e.g. answering `26` when option B is `26`).

**Why it matters:** Recovers credit when the model states the right number but not the letter. In isolation, accuracy matched the baseline because many failures are reasoning errors, not formatting.

**Example idea:** If the model outputs `26` and options are `A: 30; B: 26`, phase 2 maps to **B**.



## Phase 3 — System prompt + strict suffix

**What it does:** Adds a **system** message (“respond with exactly one letter…”) and appends **“Respond with exactly one letter (A, B, C, or D). Nothing else.”** to the user prompt.

**Why it matters:** Pushes the model toward a single-letter reply; parse rate reached 100% in our run, with moderate accuracy gains over baseline.



### Example — Number Comparison (`math_compare_39_23`)

**Gold:** A · **Predicted:** A · **Correct:** True


In [8]:

item_uid = "math_compare_39_23"
print(build_prompt({3}, item_uid)[:1200])
print()
r = load_result("phase_3.csv", item_uid)
print(r["raw_response"][:600])


Choose which is larger. Answer with A or B. A: 39; B: 23

Respond with exactly one letter (A, B, C, or D). Nothing else.

A: 39



## Phase 4 — Few-shot per trial type

**What it does:** Prepends one solved mini-example for the same `trial_type` (see `FEW_SHOT_EXAMPLES` in the script) before the actual item.

**Why it matters:** Can teach output format, but in our run it **hurt** accuracy vs baseline—possibly from distribution shift or extra length.



### Example — Number Comparison (`math_compare_8_4`)

**Gold:** B · **Predicted:** B · **Correct:** True


In [9]:

item_uid = "math_compare_8_4"
print(build_prompt({4}, item_uid)[:2000])
print()
r = load_result("phase_4.csv", item_uid)
print(r["raw_response"][:600])


Here are some solved examples:

Problem: 15, 8
Options:
A) 15
B) 8
Answer: A

Now solve this one:

Choose which is larger. Answer with A or B. A: 4; B: 8

The answer is B: 8

Explanation:

In this problem, we are comparing two numbers: 4 and 8. To determine which number is larger, we can compare them directly.

4 is less than 8.

Therefore, 8 is larger than 4.



## Phase 5 — Chain-of-thought on arithmetic

**What it does:** For `Addition`, `Subtraction`, `Multiplication`, and `Fraction`, appends: *“Think step by step… then state your final answer as a single letter.”* Increases `max_new_tokens` to 128 for these runs.

**Why it matters:** Strongest **single** improvement in our experiments: explicit reasoning helps arithmetic. Non-arithmetic types are unchanged.



### Example — Addition (`math_add_1_1`)

**Gold:** D · **Predicted:** B · **Correct:** False


In [10]:

item_uid = "math_add_1_1"
print(build_prompt({5}, item_uid)[:1500])
print()
r = load_result("phase_5.csv", item_uid)
print(r["raw_response"][:1000])


Add the numbers and choose the correct answer: 1+1. Answer with A, B, C, or D. A: 3; B: 0; C: 1; D: 2

Think step by step to compute the answer, then state your final answer as a single letter.

The correct answer is B: 0.

Here's the step-by-step process:

1. Add the numbers 1 and 1 together: 1 + 1 = 2
2. Since the sum is 2, the correct answer is B: 0.



### Example — Fraction (`math_fraction_15_15`)

**Gold:** A · **Predicted:** (none — unparsed) · **Correct:** False

CoT produced a long derivation; the parser did not extract a letter (`parsed=False`).


In [11]:
item_uid = "math_fraction_15_15"
print("--- Prompt (phase 5, includes CoT for Fraction) ---")
print(build_prompt({5}, item_uid)[:1800])
print()
r = load_result("phase_5.csv", item_uid)
print("--- Model output (truncated) ---")
print(r["raw_response"][:1200])


--- Prompt (phase 5, includes CoT for Fraction) ---
Add the numbers and choose the correct answer: 1/5+1/5. Answer with A, B, C, or D. A: 2/5; B: 1/2; C: 3/5; D: 1/10

Think step by step to compute the answer, then state your final answer as a single letter.

--- Model output (truncated) ---
To add the fractions 1/5 and 1/5, we need to find a common denominator. The least common multiple (LCM) of 5 and 5 is 5. So, we rewrite the fractions with a denominator of 5:

1/5 = 1 * 1/5 * 1 = 1/5
1/5 = 1 * 1/5 * 1 = 1/5

Now, we can add the numerators:

1/5 + 1/5 = 1 + 1 = 2

So, the sum



## Combined runs: `1+5` and `1+2+3+4+5`

- **`1+5`:** Multiline prompts plus CoT on arithmetic — strong vs baseline, but in our run **slightly below** phase 5 alone (trade-off between prompt style and CoT interaction).
- **`1+2+3+4+5`:** All toggles on — high parse rate; accuracy between the best single phases and pure phase 5.

Below: same **Number Comparison** item (`math_compare_8_4`) for the combined `1+5` prompt and CSV output.



### Example — `1+5` — Number Comparison (`math_compare_8_4`)

**Gold:** B · **Predicted:** B · **Correct:** True


In [12]:

item_uid = "math_compare_8_4"
print(build_prompt({1, 5}, item_uid)[:1500])
print()
r = load_result("phase_1_5.csv", item_uid)
print(r["raw_response"][:800])


Which number is larger: 8, 4?

A) 4
B) 8

Answer with one letter.

The answer is B) 8.

When comparing two numbers, the number with the greater value is larger. In this case, 8 is greater than 4, so the larger number is 8.



### Example — `1+2+3+4+5` — Number Comparison (`math_compare_8_4`)

**Gold:** B · **Predicted:** B · **Correct:** True


In [13]:
item_uid = "math_compare_8_4"
print("--- Prompt (phases 1+2+3+4+5) ---")
print(build_prompt({1, 2, 3, 4, 5}, item_uid)[:3500])
print()
r = load_result("phase_1_2_3_4_5.csv", item_uid)
print("--- Model output (truncated) ---")
print(r["raw_response"][:800])


--- Prompt (phases 1+2+3+4+5) ---
Here are some solved examples:

Problem: 15, 8
Options:
A) 15
B) 8
Answer: A

Now solve this one:

Which number is larger: 8, 4?

A) 4
B) 8

Answer with one letter.

Respond with exactly one letter (A, B, C, or D). Nothing else.

--- Model output (truncated) ---
B

Answer: B



---

## Full results table (exported)

Same data as `results/egma-math-phases/phase_summary.md`, reproducible below.


In [14]:

summary_df


,config,description,n,accuracy,parse_rate,delta_pp_vs_baseline
0,baseline,0 — baseline (manifest-style prompts),179,29.6%,93.9%,—
1,1,1 — multiline prompts + block options,179,50.8%,95.5%,+21.2
2,2,2 — numeric / value → label parse fallback,179,29.6%,93.9%,+0.0
3,3,3 — system prompt + strict answer suffix,179,36.9%,100.0%,+7.3
4,4,4 — few-shot examples per trial type,179,27.9%,98.9%,-1.7
5,5,5 — chain-of-thought on arithmetic types,179,57.0%,81.0%,+27.4
6,1_5,1 + 5 combined,179,53.6%,87.7%,+24.0
7,1_2_3_4_5,1 + 2 + 3 + 4 + 5 combined,179,54.2%,100.0%,+24.6
